# Kiểm chứng suy luận frozen research_v1 trên Colab

**Đây là notebook source sạch, chưa có output execution.** Nó mô tả workflow có thể chạy lại; không phải bản sao byte-for-byte của notebook đã thực thi. Các cell Python được kiểm tra AST và cấu trúc notebook được kiểm tra nbformat tại local. Không suy ra Colab PASS từ việc notebook này tồn tại.

Lượt Colab thực tế dùng staged uploads/prefetch helpers và có evidence riêng tại `results/research_colab_*`, cùng notebook Colab đã lưu trên tài khoản của người dùng. Tình trạng tải bản raw executed `.ipynb` phải xem trong hồ sơ evidence; notebook source này không thay thế nó.

Phạm vi: chạy lại suy luận trên đúng corpus/index/checkpoints đã chốt. Không train lại, không chọn hyperparameter. Dev/test đã được dùng ở local; đây là replay trên môi trường khác, không phải một holdout độc lập mới. Đánh giá faithfulness/citation do người chấm là công việc riêng.

## Trước khi chạy

Chọn runtime GPU miễn phí (ví dụ T4), không mua tài nguyên. Upload **sáu file** vào `/content`:

1. `research_source.zip`
2. `research_data.zip`
3. `research_models.zip`
4. `research_evidence.zip`
5. `compare_research_replay.py` (source từ `scripts/compare_research_replay.py`)
6. `test_research_replay_comparison.py` (source từ `tests/test_research_replay_comparison.py`)

Bốn ZIP là các gói gốc bất biến; ZIP source chưa chứa hai file audit bổ sung. Cell đầu kiểm tra SHA256 trước khi copy chúng vào project. Chờ upload hoàn tất rồi mới chạy.

Dùng runtime/thư mục sạch. Mọi xung đột nội dung được giữ nguyên và báo lỗi. Các run và evidence output không được ghi đè. Nếu phiên bị ngắt, lưu evidence hiện có và chỉ chạy tiếp các cell chưa hoàn tất; không gọi một run được tái sử dụng là lần chạy mới.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, subprocess, sys, zipfile

PROJECT_DIR = Path('/content/RAG')
EXPECTED_BUNDLE_SHA256 = {'research_source.zip': 'abbf059e4edd549a3002f5d66ca83e5e9a55bac06fba26f80adfb43338e1ba86', 'research_data.zip': '401d0462de73c97e9889a72a6385178df9406975d9080027b5147f47cad8c03b', 'research_models.zip': '4db8c33a8b09770ae204223895eddb2eba6cd11556bcc36c7e7a19a13c758fc2', 'research_evidence.zip': '3792db9430d6dfcce0c8ea762b2f41ed9c3420a59a9f5134d6e7efca90f82b47'}
SUPPLEMENTAL_FILES = {'compare_research_replay.py': {'target': 'scripts/compare_research_replay.py', 'sha256': '84c411270c5750e018bfcb4f0188f882702ff72f9356a98aab6681afc75922e4'}, 'test_research_replay_comparison.py': {'target': 'tests/test_research_replay_comparison.py', 'sha256': 'f585ae6f0b044d055d2b6002d647f51d1e07b7d187ffda49fc9f3dd2454daffd'}}

def digest(path):
    hasher = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024):
            hasher.update(block)
    return hasher.hexdigest()

def extract_bundle(path, kind):
    assert path.exists(), f'Upload {path.name} vào /content trước.'
    assert digest(path) == EXPECTED_BUNDLE_SHA256[path.name], f'Sai bundle SHA256: {path.name}'
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        assert len(names) == len(set(names)), 'Archive contains duplicate members'
        metadata = json.loads(archive.read('bundle_manifest.json'))
        assert metadata['kind'] == kind, 'Sai loại bundle'
        expected_files = metadata['files_sha256']
        assert expected_files, 'Bundle has no payload files'
        assert set(names) == {'bundle_manifest.json', *('RAG/' + name for name in expected_files)}, 'Unexpected archive members'
        # Check all payloads and existing destinations before extracting anything.
        for relative, expected in expected_files.items():
            member = 'RAG/' + relative
            target = (Path('/content') / member).resolve()
            assert target.is_relative_to(PROJECT_DIR.resolve()), 'Archive path escapes project'
            hasher = hashlib.sha256()
            with archive.open(member) as stream:
                while block := stream.read(8 * 1024 * 1024):
                    hasher.update(block)
            assert hasher.hexdigest() == expected, f'Payload checksum mismatch: {relative}'
            if target.exists():
                assert target.is_file() and digest(target) == expected, f'Giữ file hiện có khác bundle: {relative}; dùng thư mục sạch.'
        # Each bundle has its own metadata. Do not overwrite /content/bundle_manifest.json.
        for relative in expected_files:
            archive.extract('RAG/' + relative, '/content')
    print('Bundle hashes verified:', path.name, 'payload files:', len(expected_files), flush=True)

for filename, kind in (('research_source.zip', 'source'), ('research_data.zip', 'data'),
                       ('research_models.zip', 'models'), ('research_evidence.zip', 'evidence')):
    extract_bundle(Path('/content') / filename, kind)

# Preflight both supplemental files before copying either. They are audit code,
# not a replacement for any frozen ML method file.
for filename, spec in SUPPLEMENTAL_FILES.items():
    uploaded = Path('/content') / filename
    destination = PROJECT_DIR / spec['target']
    assert uploaded.is_file() and digest(uploaded) == spec['sha256'], f'Sai supplemental SHA256: {filename}'
    assert destination.resolve().is_relative_to(PROJECT_DIR.resolve()), 'Supplemental target escapes project'
    if destination.exists():
        assert destination.is_file() and digest(destination) == spec['sha256'], f'Giữ supplemental target đang xung đột: {destination}'
for filename, spec in SUPPLEMENTAL_FILES.items():
    destination = PROJECT_DIR / spec['target']
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        with (Path('/content') / filename).open('rb') as source, destination.open('xb') as output:
            while block := source.read(8 * 1024 * 1024):
                output.write(block)
    assert digest(destination) == spec['sha256']
    print('Supplemental source verified:', spec['target'], spec['sha256'], flush=True)

os.chdir(PROJECT_DIR)
lock = json.loads(Path('results/research_v1_test_lock.json').read_text(encoding='utf8'))
for name, expected in lock['source_sha256'].items():
    assert digest(Path(name)) == expected, f'Frozen method source mismatch: {name}'
print('Frozen method files match lock', flush=True)

COMMAND_LOG_DIR = Path('results/research_colab_commands')
COMMAND_LOG_DIR.mkdir(parents=True, exist_ok=True)
def run(*args):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log_path = COMMAND_LOG_DIR / (stamp + '.log')
    command = [sys.executable, '-u', *args]
    with log_path.open('x', encoding='utf8') as log:
        log.write(json.dumps({'command': command, 'cwd': str(PROJECT_DIR), 'started_utc': stamp}) + '\n')
        log.flush()
        process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
        log.write('\nEXIT_CODE=' + str(exit_code) + '\n')
    print('Command evidence:', log_path, 'exit:', exit_code, flush=True)
    if exit_code:
        raise subprocess.CalledProcessError(exit_code, command)


## Môi trường và test hợp đồng

Giữ Torch CUDA có sẵn nếu đáp ứng requirements; không ép wheel CUDA của Windows lên Colab. Phiên bản thực tế và GPU được ghi vào JSON. `bf16_api_report` chỉ là kết quả API, không phải khẳng định GPU có BF16 phần cứng. Workflow này không huấn luyện.

Nếu Colab yêu cầu restart sau cài đặt, restart rồi chạy lại cell cấu hình/input checks trước khi tiếp tục. Test research gồm cả validator audit đã kiểm tra SHA. Log có stdout/stderr và exit code, không chỉ dấu hoàn tất của giao diện.


In [ ]:
run('-m', 'pip', 'install', '-r', 'requirements/research.txt', '-r', 'requirements/test.txt')
import importlib.metadata, platform, torch
assert torch.cuda.is_available(), 'Chọn runtime GPU trước khi chạy frozen verification.'
packages = ['torch', 'transformers', 'sentence-transformers', 'datasets', 'faiss-cpu',
            'rank-bm25', 'accelerate', 'numpy', 'fastapi', 'pydantic', 'uvicorn', 'httpx']
properties = torch.cuda.get_device_properties(0)
environment = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'packages': {name: importlib.metadata.version(name) for name in packages},
    'torch_cuda_available': torch.cuda.is_available(), 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'device_capability': list(torch.cuda.get_device_capability(0)),
    'total_gpu_memory_bytes': properties.total_memory,
    'bf16_api_report': torch.cuda.is_bf16_supported(),
}
environment_path = Path('results/research_colab_environment.json')
if environment_path.exists():
    assert json.loads(environment_path.read_text(encoding='utf8')) == environment, 'Giữ environment record khác hiện có; dùng evidence path/runtime mới.'
else:
    with environment_path.open('x', encoding='utf8') as stream:
        json.dump(environment, stream, indent=2)
        stream.write('\n')
print(json.dumps(environment, indent=2))
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_research*.py', '-v')
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_triviaqa_metrics.py', '-v')


## Kiểm tra corpus/index đóng băng

Kiểm tra checksums, cấu hình index và vectors trên corpus đã upload. Không dựng lại corpus, lọc câu khó hoặc thay nhãn trong workflow frozen này.


In [ ]:
DATA_DIR = Path('data/research_v1')
run('-m', 'scripts.check_research', '--data-dir', str(DATA_DIR))
print(json.dumps(json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf8'))['counts'], indent=2))


## Kiểm tra hai checkpoint riêng

Hai model phải complete, đúng vai trò và khớp dataset/model hashes. `--download-base` cho phép tải đúng base revision công khai để kiểm tra cập nhật trọng số; bước này không train hoặc chọn lại checkpoint.


In [ ]:
for system in ('closed_book', 'rag'):
    run('-m', 'scripts.check_research', '--data-dir', str(DATA_DIR),
        '--model-run', f'checkpoints/research_v1/{system}', '--download-base')


## Replay đủ 200 dev

Chạy nguyên năm arms và tính lại báo cáo. Thư mục run phải mới; không dùng điều kiện “đã tồn tại thì bỏ qua” để biến artifact cũ thành bằng chứng cho execution mới. Không thay retrieval/generation parameters sau khi xem kết quả.


In [ ]:
DEV_RUN = Path('results/research_colab_dev')
assert not DEV_RUN.exists(), f'Giữ run hiện có: {DEV_RUN}; dùng runtime sạch hoặc ghi rõ một đường dẫn run mới.'
run('-m', 'experiments.research_evaluate', '--data-dir', str(DATA_DIR),
    '--split', 'dev', '--run-dir', str(DEV_RUN))
dev_manifest = json.loads((DEV_RUN / 'manifest.json').read_text(encoding='utf8'))
assert dev_manifest['status'] == 'complete' and dev_manifest['n'] == 200
run('-m', 'scripts.report_research', '--run-dir', str(DEV_RUN))
print(json.dumps({'split': dev_manifest['split'], 'status': dev_manifest['status'], 'n': dev_manifest['n']}, indent=2))


## Replay đủ 300 test đã được chốt ở local

Bộ test này đã được sử dụng. Cờ `--final` thực thi protocol frozen và đủ số câu; không làm lượt replay thành một holdout mới. Cùng năm arms, không lấy subset và không tinh chỉnh theo test.


In [ ]:
TEST_RUN = Path('results/research_colab_test')
assert not TEST_RUN.exists(), f'Giữ run hiện có: {TEST_RUN}; dùng runtime sạch hoặc ghi rõ một đường dẫn run mới.'
run('-m', 'experiments.research_evaluate', '--data-dir', str(DATA_DIR),
    '--split', 'test', '--final', '--run-dir', str(TEST_RUN))
test_manifest = json.loads((TEST_RUN / 'manifest.json').read_text(encoding='utf8'))
assert test_manifest['status'] == 'complete' and test_manifest['n'] == 300
run('-m', 'scripts.report_research', '--run-dir', str(TEST_RUN))
print(json.dumps({'split': test_manifest['split'], 'status': test_manifest['status'], 'n': test_manifest['n']}, indent=2))


## Đối chiếu với artifact local trong evidence ZIP

Validator bổ sung kiểm tra SHA predictions/snapshot của từng run, complete manifests, đúng 200/300 dòng, ID duy nhất, đúng năm arms, QA records và source/data/model identities theo lock. Archive snapshot của hai môi trường có thể khác; bốn method hashes phải khớp.

`PASS_EXACT_ANSWERS` yêu cầu toàn bộ raw answers khớp và QA EM/F1 hữu hạn khớp trong 1e-12. Sai khác hợp lệ được giữ là `REVIEW_REQUIRED`. Provenance không hợp lệ được ghi `INVALID_PROVENANCE` và lệnh trả exit code khác 0. Không nới tiêu chí hoặc sửa predictions để lấy PASS.

Ranking/context là diagnostics riêng; answer PASS không có nghĩa mọi retrieval score hoặc full ranking bitwise giống nhau. Tóm tắt phía dưới đọc trạng thái thực từ JSON, không hardcode kết quả thành công.


In [ ]:
COMPARISON_PATH = Path('results/research_colab_reproduction_comparison.json')
assert not COMPARISON_PATH.exists(), f'Giữ comparison hiện có: {COMPARISON_PATH}'
run('-m', 'scripts.compare_research_replay',
    '--original-dev', 'results/research_v1_dev',
    '--original-test', 'results/research_v1_test',
    '--replay-dev', str(DEV_RUN), '--replay-test', str(TEST_RUN),
    '--lock', 'results/research_v1_test_lock.json',
    '--output', str(COMPARISON_PATH))
comparison = json.loads(COMPARISON_PATH.read_text(encoding='utf8'))
summary = {'status': comparison['status'], 'scope': comparison['scope'], 'splits': {}}
for split, item in comparison['comparisons'].items():
    summary['splits'][split] = {
        'status': item['status'], 'n': item['n'], 'expected_n': item['expected_n'],
        'raw_answer_mismatches': {arm: len(ids) for arm, ids in item['raw_answer_mismatch_ids'].items()},
        'qa_metrics_equal_within_tolerance': item['qa_metrics_equal_within_tolerance'],
        'retrieval_context_diagnostic_status': item['evidence_diagnostics']['status'],
        'ranking_mismatch_ids': item['evidence_diagnostics']['ranking_mismatch_ids'],
    }
print(json.dumps(summary, indent=2, ensure_ascii=False))


## Lưu bằng chứng thực chạy

Gói evidence chứa manifests, predictions nén, metrics, báo cáo tính lại, environment, command logs và comparison. Tải ZIP từ Files trước khi VM kết thúc. Lưu riêng hai file audit có SHA và notebook đã có outputs; exporter source gốc không tự chứa raw executed notebook.

Notebook source hiện tại vẫn là một workflow sạch, không phải raw executed copy của lượt Colab trước. Lượt Colab thực tế có thể chạy staged uploads/prefetch để tiết kiệm thời gian; chỉ artifacts/logs của execution đó chứng minh kết quả của nó.

Phiếu `human_review_blinded.csv` cần người chấm và rationale theo `docs/REVIEW_RUBRIC.md`. Không gọi phiếu trống hoặc `agent_self_review` là human evaluation. Workflow này cũng không chứng minh tái lập huấn luyện.


In [ ]:
EVIDENCE_PATH = Path('outputs/research_colab_evidence.zip')
assert not EVIDENCE_PATH.exists(), f'Giữ evidence bundle hiện có: {EVIDENCE_PATH}'
run('-m', 'scripts.export_research_bundle', '--kind', 'evidence', '--data-dir', str(DATA_DIR),
    '--output', str(EVIDENCE_PATH))
print(json.dumps({'evidence_path': str(PROJECT_DIR / EVIDENCE_PATH),
                  'bytes': EVIDENCE_PATH.stat().st_size, 'sha256': digest(EVIDENCE_PATH),
                  'comparison_status': json.loads(COMPARISON_PATH.read_text(encoding='utf8'))['status']}, indent=2))
